# v133b_small_tal_f2 — Small-TAL (`max_boost=4.0`) honest Fold-2 OOF 学習・検証 (train=(0, 1), val=fold-2 非接触)

- `v130` pilot 実験で **`+0.01414` PQ（`0.43245 -> 0.44660`, 95% CI `[+0.00509, +0.02319]`, `P(>0)=99.75%`, 小個体 recall `+6.08 pt`）** を記録した `Small-Instance TAL Equalization` を、`prod` (`yolo_fold_2`) / `v96-full` と **100% 同一の `single_cls=True` レシピ**（変更は `PureSmallTALSegmentationLoss(max_boost=4.0)` の 1 点のみ）で検証・学習する。
- データセットは Colab ローカル NVMe (`/content/solar_v133b_small_tal_f2/yolo_datasets`) に構築して高速 I/O で学習し、重みと manifest は Google Drive (`runs/v133_small_tal/`, `artifacts/`) に永続保存する。
- `best.pt`・`last.pt` の単独 PQ に加え、既にメモリ上にある `prod` キャッシュとの **1-to-1 2 検出器連続融合 (`prod + small_tal`)** および 2,000 回 paired bootstrap 95% CI を同時算出する。


In [ ]:
# [1/6] 依存・Drive・設定
%pip install -q "ultralytics==8.4.120" pycocotools
import hashlib, json, math, os, pickle, random, shutil, sys, tempfile, time
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

class CFG:
    SEED = 3407
    FOLD = 2
    ARMS = ('small_tal',)
    TRAIN_FOLDS = (0, 1)
    IS_ALL_DATA = False
    MAX_BOOST = 4.0
    GROUPED_FOLDS_SHA256 = "febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e"
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    RUNS_ROOT = DRIVE_ROOT / "runs" / "v133_small_tal"
    CACHE_OUT_DIR = ARTIFACT_ROOT / "v133_oof_caches"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v133b_small_tal_f2")
    YOLO_ROOT = LOCAL_ROOT / "yolo_datasets"
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    PROD_YOLO = DRIVE_ROOT / "runs" / f"yolo_fold_{FOLD}" / "weights" / "best.pt"
    REFINER_CKPT = ARTIFACT_ROOT / f"mediseg_edgeattnet_refiner_fold{FOLD}_resume.pt"
    MANIFEST = ARTIFACT_ROOT / "v133b_small_tal_f2_colab_manifest.json"
    YOLO_MODEL = "yolo11m-seg.pt"
    YOLO_IMGSZ = 1280
    YOLO_BATCH = 4
    YOLO_EPOCHS = 70
    YOLO_PATIENCE = 14
    YOLO_MASK_RATIO = 2
    YOLO_CLS = 0.5
    PREDICT_CONF_FLOOR = 0.08
    DIAG_CONF_FLOOR = 0.02
    PREDICT_IOU = 0.55
    CONF_SWEEP = (0.20, 0.24, 0.26, 0.28, 0.32)
    PRIMARY_CONF = 0.26
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    BASELINE_PQ = 0.444238
    BOOTSTRAP_ITERS = 2000
    SPARSE_R = 0.77

for path in (CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.YOLO_ROOT, CFG.RUNS_ROOT, CFG.CACHE_OUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if not CFG.REFINER_CKPT.exists():
    _cands = sorted(CFG.ARTIFACT_ROOT.glob(f"*refiner*fold{CFG.FOLD}*.pt"))
    if _cands:
        CFG.REFINER_CKPT = _cands[0]
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
required = {"competition data": CFG.DATA_CACHE, "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
            "前処理画像": CFG.PREPROC_CACHE, "prod YOLO": CFG.PROD_YOLO, "EdgeAtt refiner": CFG.REFINER_CKPT}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提 artifact が見つからない: {missing}")
assert torch.cuda.is_available(), "GPU ランタイムを選択してください"
RUNTIME_INFO = {"started_utc": datetime.now(timezone.utc).isoformat(), "GPU": torch.cuda.get_device_name(0),
                "python": sys.version, "torch": torch.__version__, "ultralytics": version("ultralytics"),
                "pycocotools": version("pycocotools"), "opencv": cv2.__version__, "numpy": np.__version__}
print(RUNTIME_INFO)


In [ ]:
# [2/6] COCO・凍結 fold (SHA256 検証)・前処理画像
import csv, io, re

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

TRAIN_JSON = next(Path(CFG.DATA_CACHE).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
coco = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

folds_path = CFG.ARTIFACT_ROOT / "grouped_folds.csv"
folds_sha256 = file_sha256(folds_path)
if folds_sha256 != CFG.GROUPED_FOLDS_SHA256:
    raise ValueError(f"grouped_folds.csv の SHA256 が不一致: {folds_sha256}")
fold_frame = pd.read_csv(folds_path)
if fold_frame["file_name"].duplicated().any() or set(fold_frame["file_name"]) != set(train_files):
    raise ValueError("fold CSV が全観測を一意にカバーしていない")
fold_of_file = dict(zip(fold_frame["file_name"], fold_frame["fold"].astype(int)))
valid_files = [n for n in train_files if fold_of_file[n] == CFG.FOLD]
fit_files = [n for n in train_files if fold_of_file[n] in CFG.TRAIN_FOLDS]

for name in tqdm(train_files, desc="前処理画像をローカルへ"):
    local = CFG.PREPROC_DIR / name
    if not local.exists():
        shutil.copy2(CFG.PREPROC_CACHE / name, local)

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

gt_rles = {image_id: [to_rle(a, int(info["height"]), int(info["width"]))
                      for a in annotations[image_id] if not a.get("iscrowd", 0)]
           for image_id, info in images.items()}
print({"観測": len(train_files), "fit": len(fit_files), "val(OOF)": len(valid_files),
       "folds_sha256": folds_sha256[:16]})


In [ ]:
# [3/6] PureSmallTALSegmentationTrainer 組込み + prod 完全準拠データセット構築・学習
# --- BEGIN spine_yolo.py ---
"""Spine aux-teacher — ultralytics integration glue & trainer (train-time + Arm B+ rescue).

Design (prereg 2026-11-13): spine polylines travel through the standard
YOLO-seg label/transform pipeline as **pseudo-instances of a reserved
class** so every geometric transform (mosaic concat, RandomPerspective
``apply_segments`` affine, flips, letterbox) applies identical math to
spine and mask. Immediately before the final ``Format`` transform a
``SplitSpineSegments`` step removes the pseudo-instances and stores the
transformed polylines on ``labels['spine_segments_px']``; the detection
pipeline then sees only real instances (model stays nc=1).

Constraints verified against ultralytics 8.4.118–8.4.120 source:
- ``verify_image_label`` asserts ``max_cls < num_cls`` -> the dataset
  yaml for spine training must declare ``nc: 2`` ({0: filament,
  1: spine_aux}); ``single_cls`` MUST be ``False`` (else ``BaseDataset``
  overwrites ``cls=1`` to ``0`` before transforms), and the custom trainer
  forces ``model.nc = 1`` and ``model.names = {0: 'filament'}``.
- ``Format.apply_instances`` calls ``cls.squeeze(1)`` -> ``labels['cls']``
  after ``SplitSpineSegments`` must have 2D shape ``(N_keep, 1)``.
- ``YOLODataset.collate_fn`` uses ``zip(*values)``, producing a ``tuple``
  of length ``B`` for custom keys -> ``split_spine_batch`` accepts both
  ``list`` and ``tuple``.
- ``SpineAuxLoss.call`` computes BCE in ``float32`` via log-sigmoid so
  AMP ``float16`` never underflows ``1 - 1e-7 -> 1.0`` into ``NaN`` grads.
"""


from collections.abc import Sequence

import numpy as np

SPINE_CLS = 1  # reserved pseudo-class in the spine-augmented dataset yaml


def spine_label_row(spine_xy: np.ndarray, img_h: int, img_w: int) -> str:
    """Format one annotation spine as a YOLO-seg label row of SPINE_CLS.

    ``spine_xy``: (N,2) image-space polyline. Output is the normalized
    ``cls x1 y1 x2 y2 ...`` polygon-format row; a polyline is a valid
    (degenerate) polygon for the transform math we rely on.
    """
    pts = np.asarray(spine_xy, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 2:
        raise ValueError("spine polyline needs >= 2 points")
    norm = pts.copy()
    norm[:, 0] = np.clip(norm[:, 0] / img_w, 0.0, 1.0)
    norm[:, 1] = np.clip(norm[:, 1] / img_h, 0.0, 1.0)
    flat = " ".join(f"{v:.6f}" for v in norm.reshape(-1))
    return f"{SPINE_CLS} {flat}"


def _ensure_min_three_points(pts: np.ndarray) -> np.ndarray:
    """Subdivide a 2-point segment by its midpoint so YOLO-seg sees >=7 tokens (not a 5-token bbox)."""
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if len(pts) == 2:
        mid = 0.5 * (pts[0] + pts[1])
        return np.vstack([pts[0], mid, pts[1]])
    return pts


def emit_label_txt(
    instances: list[np.ndarray],
    spines: list[np.ndarray],
    img_h: int,
    img_w: int,
    real_cls: int = 0,
) -> str:
    """Emit a full label .txt: real instance polygons + spine pseudo-rows.

    ``instances``: list of (M,2) polygon point arrays (image coords).
    ``spines``: list of (N,2) spine polylines, same order as instances.
    2-point spines are midpoint-subdivided to 3 collinear points so
    Ultralytics ``verify_image_label`` never rejects them as 5-column boxes.
    """
    rows = []
    for poly in instances:
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        if len(pts) < 3:
            continue
        norm = pts.copy()
        norm[:, 0] = np.clip(norm[:, 0] / img_w, 0.0, 1.0)
        norm[:, 1] = np.clip(norm[:, 1] / img_h, 0.0, 1.0)
        rows.append(f"{real_cls} " + " ".join(f"{v:.6f}" for v in norm.reshape(-1)))
    for spine in spines:
        pts = np.asarray(spine, dtype=np.float64).reshape(-1, 2)
        if len(pts) >= 2:
            rows.append(spine_label_row(_ensure_min_three_points(pts), img_h, img_w))
    return "\n".join(rows) + ("\n" if rows else "")


def _pad_lrtb_from_labels(labels: dict) -> tuple[int, int, int, int]:
    """Extract (left, top, right, bottom) canvas padding pixels from LetterBox metadata."""
    ratio_pad = labels.get("ratio_pad")
    if not isinstance(ratio_pad, (list, tuple)) or len(ratio_pad) < 2:
        return (0, 0, 0, 0)
    pad_wh = ratio_pad[1]
    if not isinstance(pad_wh, (list, tuple)) or len(pad_wh) < 2:
        return (0, 0, 0, 0)
    dw, dh = float(pad_wh[0]), float(pad_wh[1])
    left = max(0, round(dw - 0.1))
    right = max(0, round(dw + 0.1))
    top = max(0, round(dh - 0.1))
    bottom = max(0, round(dh + 0.1))
    return (left, top, right, bottom)


class SplitSpineSegments:
    """Pre-Format transform: move SPINE_CLS instances to spine_segments_px.

    Sits last in the geometric Compose (inserted just before ``Format``).
    Preserves ``cls`` as 2D ``(N_keep, 1)`` when input is 2D or ``img`` is
    present (required by ``Format.apply_instances`` ``cls.squeeze(1)``), and
    converts normalized segments to canvas pixel coordinates when
    ``augment=False`` (e.g. validation or ``close_mosaic`` epochs).
    """

    def __init__(self, spine_cls: int = SPINE_CLS):
        self.spine_cls = spine_cls

    def __call__(self, labels: dict) -> dict:
        instances = labels.get("instances")
        cls = labels.get("cls")
        labels["spine_pad_lrtb"] = _pad_lrtb_from_labels(labels)
        if instances is None or cls is None:
            labels["spine_segments_px"] = []
            return labels
        cls_arr = np.asarray(cls)
        keep_2d = cls_arr.ndim == 2 or "img" in labels
        flat_cls = cls_arr.reshape(-1)
        is_spine = flat_cls == self.spine_cls
        segs = np.asarray(instances.segments)
        if segs.ndim == 3 and "img" in labels and getattr(instances, "normalized", False):
            h, w = labels["img"].shape[:2]
            scale = np.array([float(w), float(h)], dtype=segs.dtype)
            segs = segs * scale
        labels["spine_segments_px"] = (
            [segs[i] for i in np.flatnonzero(is_spine)] if segs.ndim == 3 else []
        )
        keep = ~is_spine
        labels["instances"] = instances[keep]
        kept_cls = flat_cls[keep]
        labels["cls"] = kept_cls.reshape(-1, 1) if keep_2d else kept_cls
        return labels


def split_spine_batch(batch: dict, spine_key: str = "spine_segments_px") -> list[list[np.ndarray]]:
    """Collect per-image spine polyline lists from a collated batch.

    ``YOLODataset.collate_fn`` aggregates unknown keys via ``zip(*values)``,
    producing a ``tuple`` of length ``B``; direct callers may pass a ``list``.
    """
    value = batch.get(spine_key, ())
    if isinstance(value, (list, tuple)):
        return [list(v) if isinstance(v, (list, tuple)) else [v] for v in value]
    return [[value]] if len(value) else [[]]


def split_pad_batch(
    batch: dict,
    batch_size: int,
    pad_key: str = "spine_pad_lrtb",
) -> list[tuple[int, int, int, int]]:
    """Extract per-image (left, top, right, bottom) padding tuples from a batch."""
    raw = batch.get(pad_key)
    if isinstance(raw, (list, tuple)) and len(raw) == batch_size:
        out: list[tuple[int, int, int, int]] = []
        for item in raw:
            if isinstance(item, (list, tuple)) and len(item) == 4:
                out.append((int(item[0]), int(item[1]), int(item[2]), int(item[3])))
            else:
                out.append((0, 0, 0, 0))
        return out
    return [(0, 0, 0, 0)] * batch_size


def build_p3_targets(
    spine_lists: list[list[np.ndarray]],
    batch_shape_hw: tuple[int, int],
    grid: int | tuple[int, int] = 160,
    radius_cells: int = 1,
    pads: Sequence[tuple[int, int, int, int]] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Rasterize each batch image's spine polylines to (pos, valid) grids."""
    from solar_filament.pipeline.spine_aux import (
        build_spine_target,
        padding_exclusion_mask,
    )

    out_h, out_w = (grid, grid) if isinstance(grid, int) else (int(grid[0]), int(grid[1]))
    h, w = int(batch_shape_hw[0]), int(batch_shape_hw[1])
    bsz = len(spine_lists)
    pos = np.zeros((bsz, out_h, out_w), dtype=bool)
    valid = np.ones_like(pos)
    for i, spines in enumerate(spine_lists):
        pad = pads[i] if pads is not None and i < len(pads) else (0, 0, 0, 0)
        vmask = (
            padding_exclusion_mask(out_h, out_w, h, w, pad=pad)
            if any(pad)
            else None
        )
        p, v = build_spine_target(
            spines,
            out_h=out_h,
            out_w=out_w,
            img_h=h,
            img_w=w,
            radius_cells=radius_cells,
            valid_mask=vmask,
        )
        pos[i], valid[i] = p, v
    return pos, valid


class SpineAuxLoss:
    """AMP-safe torch implementation of positive/negative separately-averaged BCE."""

    @staticmethod
    def call(logits, pos, valid, eps: float = 1e-7):
        import torch
        import torch.nn.functional as F

        del eps  # log-sigmoid BCE is exact and avoids float16 clamp rounding to 1.0
        logits32 = logits.float()
        pos_t = torch.as_tensor(pos, dtype=torch.bool, device=logits.device)
        valid_t = torch.as_tensor(valid, dtype=torch.bool, device=logits.device)
        if pos_t.ndim == 3 and logits32.ndim == 4:
            pos_t = pos_t.unsqueeze(1)
            valid_t = valid_t.unsqueeze(1)
        pos_sel = pos_t & valid_t
        neg_sel = (~pos_t) & valid_t
        zero = logits32.sum() * 0.0
        pos_term = (
            F.binary_cross_entropy_with_logits(
                logits32[pos_sel], torch.ones_like(logits32[pos_sel]), reduction="mean"
            )
            if pos_sel.any()
            else zero
        )
        neg_term = (
            F.binary_cross_entropy_with_logits(
                logits32[neg_sel], torch.zeros_like(logits32[neg_sel]), reduction="mean"
            )
            if neg_sel.any()
            else zero
        )
        return pos_term + neg_term


def compute_spine_ridge_score(
    mask: np.ndarray,
    spine_prob: np.ndarray,
    *,
    top_quantile: float = 0.75,
) -> float:
    """Mean spine heatmap probability along the top-quantile ridge inside ``mask``.

    Maps pixel coordinates directly onto the P3 ``spine_prob`` grid ``(gh, gw)``
    without upsampling ``spine_prob`` to ``2048x2048``.
    """
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return 0.0
    h, w = mask.shape
    gh, gw = spine_prob.shape
    cy = np.clip((ys.astype(np.int64) * gh) // h, 0, gh - 1)
    cx = np.clip((xs.astype(np.int64) * gw) // w, 0, gw - 1)
    vals = spine_prob[cy, cx].astype(np.float64)
    if len(vals) <= 4 or top_quantile <= 0.0:
        return float(vals.mean())
    thr = float(np.quantile(vals, top_quantile))
    top_vals = vals[vals >= thr]
    return float(top_vals.mean()) if len(top_vals) else float(vals.mean())


def selective_spine_rescue(
    conf: Sequence[float] | np.ndarray,
    areas: Sequence[float] | np.ndarray,
    spine_scores: Sequence[float] | np.ndarray,
    *,
    primary_conf: float = 0.26,
    rescue_floor: float = 0.14,
    min_spine: float = 0.65,
    min_area: int = 20,
) -> list[int]:
    """Keep all primary proposals plus low-conf proposals with strong P3 spine ridge support."""
    conf_arr = np.asarray(conf, dtype=np.float64)
    area_arr = np.asarray(areas, dtype=np.float64)
    spine_arr = np.asarray(spine_scores, dtype=np.float64)
    if not (len(conf_arr) == len(area_arr) == len(spine_arr)):
        raise ValueError("conf, areas, and spine_scores must have equal length")
    kept: list[int] = []
    for i in range(len(conf_arr)):
        if area_arr[i] < min_area:
            continue
        if conf_arr[i] >= primary_conf or (
            conf_arr[i] >= rescue_floor and spine_arr[i] >= min_spine
        ):
            kept.append(i)
    return kept


def build_spine_trainer_classes(aux_weight: float = 0.0, radius_cells: int = 1):
    """Build Ultralytics Dataset, Loss, and SegmentationTrainer classes for Spine A/B.

    Both Arm A (``aux_weight=0.0``) and Arm B (``aux_weight>0.0``) can share the
    exact same dataset/transform/trainer path so the ONLY difference between
    Arm A and Arm B is the P3 ``1x1 conv`` spine auxiliary supervision.
    """
    from copy import copy

    from torch import nn
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.models.yolo.segment import SegmentationTrainer
    from ultralytics.nn.tasks import SegmentationModel
    from ultralytics.utils import DEFAULT_CFG, RANK
    from ultralytics.utils.loss import v8SegmentationLoss
    from ultralytics.utils.torch_utils import unwrap_model

    class SpineYOLODataset(YOLODataset):
        def build_transforms(self, hyp=None):
            transforms = super().build_transforms(hyp)
            transforms.insert(-1, SplitSpineSegments(spine_cls=SPINE_CLS))
            return transforms

    class SpineSegmentationLoss(v8SegmentationLoss):
        def __init__(self, model: nn.Module, tal_topk: int = 10):
            super().__init__(model, tal_topk=tal_topk)
            self.spine_head = getattr(model, "spine_head", None)
            self.aux_weight = float(aux_weight)
            self.radius_cells = int(radius_cells)

        def loss(self, preds: dict, batch: dict):
            loss_vec, loss_items = super().loss(preds, batch)
            if (
                self.aux_weight > 0.0
                and self.spine_head is not None
                and "spine_segments_px" in batch
            ):
                p3_feat = preds["feats"][0]
                aux_logits = self.spine_head(p3_feat)
                bsz = int(aux_logits.shape[0])
                spine_lists = split_spine_batch(batch)
                if len(spine_lists) == bsz:
                    pads = split_pad_batch(batch, bsz)
                    pos, valid = build_p3_targets(
                        spine_lists,
                        batch["img"].shape[-2:],
                        grid=aux_logits.shape[-2:],
                        radius_cells=self.radius_cells,
                        pads=pads,
                    )
                    aux_raw = SpineAuxLoss.call(aux_logits, pos, valid)
                    weighted = self.aux_weight * aux_raw
                    loss_vec[4] = loss_vec[4] + weighted * bsz
                    loss_items["sem_loss"] = (loss_items["sem_loss"] + weighted).detach()
            return loss_vec, loss_items

    class SpineSegmentationTrainer(SegmentationTrainer):
        def __init__(self, cfg=DEFAULT_CFG, overrides=None, _callbacks=None):
            overrides = dict(overrides or {})
            # Never allow BaseDataset.update_labels to overwrite SPINE_CLS=1 -> 0
            overrides["single_cls"] = False
            super().__init__(cfg=cfg, overrides=overrides, _callbacks=_callbacks)

        def get_model(self, cfg=None, weights=None, verbose: bool = True):
            # Force single real output class (0: filament) regardless of data.yaml nc=2
            orig_names = self.data.get("names")
            self.data = {**self.data, "names": {0: "filament"}}
            try:
                model = self.set_model_names_for_load(
                    SegmentationModel(
                        cfg,
                        nc=1,
                        ch=self.data.get("channels", 3),
                        verbose=verbose and RANK == -1,
                    )
                )
                if weights:
                    model.load(weights)
            finally:
                if orig_names is not None:
                    self.data["names"] = orig_names
            model.nc = 1
            model.names = {0: "filament"}
            if aux_weight > 0.0:
                seg_head = model.model[-1]
                p3_ch = int(seg_head.cv2[0][0].conv.in_channels)
                model.spine_head = nn.Conv2d(p3_ch, 1, kernel_size=1)
            return model

        def set_model_attributes(self):
            super().set_model_attributes()
            umodel = unwrap_model(self.model)
            umodel.nc = 1
            umodel.names = {0: "filament"}
            umodel.criterion = SpineSegmentationLoss(umodel)

        def get_class_counts(self):
            classes = np.concatenate(
                [lb["cls"].reshape(-1) for lb in self.train_loader.dataset.labels], 0
            )
            real = classes[classes == 0].astype(int)
            return np.bincount(real, minlength=1).astype(np.float32)

        def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
            gs = max(int(unwrap_model(self.model).stride.max()), 32)
            return SpineYOLODataset(
                img_path=img_path,
                imgsz=self.args.imgsz,
                batch_size=batch,
                augment=mode == "train",
                hyp=self.args,
                rect=self.args.rect or mode == "val",
                cache=self.args.cache or None,
                single_cls=False,
                stride=gs,
                pad=0.0 if mode == "train" else 0.5,
                prefix=f"{mode}: ",
                task=self.args.task,
                classes=self.args.classes,
                data=self.data,
                fraction=self.args.fraction if mode == "train" else 1.0,
            )

        def get_validator(self):
            val_args = copy(self.args)
            val_args.single_cls = True
            from ultralytics.models import yolo
            return yolo.segment.SegmentationValidator(
                self.test_loader,
                save_dir=self.save_dir,
                args=val_args,
                _callbacks=self.callbacks,
            )

    return SpineYOLODataset, SpineSegmentationLoss, SpineSegmentationTrainer


def extract_p3_spine_prob(yolo_obj, image_path: str, *, imgsz: int = 1280, conf: float = 0.02, iou: float = 0.55):
    """Run standard ``YOLO.predict`` while capturing P3 ``spine_head`` probability map ``(H_p3, W_p3)``.

    Returns ``(result, spine_prob_or_none)``. If ``yolo_obj.model`` has no ``spine_head``
    (e.g. Arm A), ``spine_prob_or_none`` is ``None`` and prediction is identical to standard YOLO.
    """
    import torch

    net = yolo_obj.model
    spine_head = getattr(net, "spine_head", None)
    if spine_head is None:
        res = yolo_obj.predict(
            str(image_path),
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            agnostic_nms=True,
            retina_masks=True,
            verbose=False,
        )[0]
        return res, None

    p3_layer_idx = int(net.model[-1].f[0])
    captured: list[torch.Tensor] = []

    def _hook(_module, _inp, out):
        captured.append(out.detach())

    handle = net.model[p3_layer_idx].register_forward_hook(_hook)
    try:
        res = yolo_obj.predict(
            str(image_path),
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            agnostic_nms=True,
            retina_masks=True,
            verbose=False,
        )[0]
        if not captured:
            return res, None
        with torch.no_grad():
            feat = captured[-1]
            spine_head = spine_head.to(device=feat.device, dtype=feat.dtype).eval()
            prob = torch.sigmoid(spine_head(feat).float())[0, 0].cpu().numpy()
        return res, prob
    finally:
        handle.remove()
# --- END spine_yolo.py ---

# --- BEGIN small_tal_yolo.py ---
"""Small-instance Task-Aligned Assigner anchor equalization for YOLO11m-seg.

Implements Idea 6 (v130):
In Ultralytics ``v8SegmentationLoss``, ``TaskAlignedAssigner(topk=10)`` assigns 10
positive anchors to large filaments, whereas small filaments (``area <= 1024 px^2``
at ``2048``, ``<= 400 px^2`` at ``1280``, 42.4% of MAGFiLO GT) physically cover only
1–3 stride-8 anchor centers inside their bounding box. Summing over positive anchors
starves small filaments by 5x–10x in ``cls_loss``, ``box_loss``, ``dfl_loss``, and
``seg_loss`` simultaneously.

``compute_anchor_equalization_weights`` equalizes positive anchor weights by
``clamp(topk / n_assigned_to_gt, 1.0, max_boost)`` and normalizes across the batch's
positive anchors so:
  1. Total positive ``target_scores`` weight sum is 100% preserved (negative anchor
     calibration at ``conf = 0.26`` is invariant).
  2. Total positive ``seg_loss`` weight sum ``== fg_mask.sum()`` (effective mask LR
     is invariant).
  3. When ``max_boost == 1.0``, all weights equal ``1.0`` (exact identity with vanilla
     Ultralytics ``v8SegmentationLoss``).
"""


import torch



def compute_anchor_equalization_weights(
    fg_mask: torch.Tensor,
    target_gt_idx: torch.Tensor,
    target_scores: torch.Tensor | None = None,
    *,
    topk: int = 10,
    max_boost: float = 4.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute mass-preserving ``(w_cls_box, w_seg)`` of shape ``(B, N_anchors)``.

    - Background anchors (``~fg_mask``) always receive weight ``1.0``.
    - Positive anchors assigned to a GT instance that received ``c`` anchors get raw
      boost ``clamp(topk / c, 1.0, max_boost)``.
    - ``w_seg[fg_mask]`` is normalized so ``w_seg[fg_mask].sum() == fg_mask.sum()``.
    - ``w_cls_box[fg_mask]`` is normalized so
      ``(target_scores * w_cls_box[..., None]).sum() == target_scores.sum()``,
      keeping the positive/negative classification balance strictly invariant.
    """
    bsz, _ = fg_mask.shape
    w_seg = torch.ones_like(fg_mask, dtype=torch.float32)
    w_cls = torch.ones_like(fg_mask, dtype=torch.float32)
    if max_boost <= 1.0 or not fg_mask.any():
        return w_cls, w_seg

    raw_fg_list: list[torch.Tensor] = []
    for b in range(bsz):
        fg_b = fg_mask[b]
        if not fg_b.any():
            continue
        gt_idx_b = target_gt_idx[b, fg_b].long()
        counts = torch.bincount(gt_idx_b, minlength=int(gt_idx_b.max().item()) + 1).float()
        per_anchor_cnt = counts[gt_idx_b].clamp(min=1.0)
        raw_b = (float(topk) / per_anchor_cnt).clamp(min=1.0, max=float(max_boost))
        raw_fg_list.append(raw_b)

    raw_fg = torch.cat(raw_fg_list, dim=0)
    fg_count = float(fg_mask.sum().item())
    norm_seg = raw_fg * (fg_count / raw_fg.sum().clamp(min=1e-6))
    w_seg[fg_mask] = norm_seg

    if target_scores is not None:
        ts_fg = target_scores[fg_mask].float().sum(dim=-1)  # (N_fg,)
        orig_ts_sum = ts_fg.sum()
        if orig_ts_sum > 1e-6:
            weighted_ts_sum = (ts_fg * raw_fg).sum().clamp(min=1e-6)
            norm_cls = raw_fg * (orig_ts_sum / weighted_ts_sum)
        else:
            norm_cls = norm_seg
        w_cls[fg_mask] = norm_cls
    else:
        w_cls[fg_mask] = norm_seg

    return w_cls, w_seg


def build_small_tal_trainer_classes(
    *,
    max_boost: float = 4.0,
    aux_weight: float = 0.0,
    radius_cells: int = 1,
):
    """Build Ultralytics Dataset, Anchor-Equalized Loss, and SegmentationTrainer classes."""
    from torch import nn
    from ultralytics.utils.ops import xyxy2xywh
    from ultralytics.utils.tal import make_anchors
    from ultralytics.utils.torch_utils import unwrap_model

    base_ds_cls, base_loss_cls, base_trainer_cls = build_spine_trainer_classes(
        aux_weight=aux_weight,
        radius_cells=radius_cells,
    )

    class SmallTALSegmentationLoss(base_loss_cls):
        def __init__(self, model: nn.Module, tal_topk: int = 10):
            super().__init__(model, tal_topk=tal_topk)
            self.tal_topk = int(tal_topk)
            self.max_boost = float(max_boost)
            self._last_w_seg: torch.Tensor | None = None

        def get_assigned_targets_and_loss(self, preds: dict, batch: dict) -> tuple:
            loss = torch.zeros(3, device=self.device)  # box, cls, dfl
            pred_distri, pred_scores = (
                preds["boxes"].permute(0, 2, 1).contiguous(),
                preds["scores"].permute(0, 2, 1).contiguous(),
            )
            anchor_points, stride_tensor = make_anchors(preds["feats"], self.stride, 0.5)

            dtype = pred_scores.dtype
            batch_size = pred_scores.shape[0]
            imgsz = (
                torch.tensor(preds["feats"][0].shape[2:], device=self.device, dtype=dtype)
                * self.stride[0]
            )

            targets = torch.cat(
                (batch["batch_idx"].view(-1, 1), batch["cls"].view(-1, 1), batch["bboxes"]), 1
            )
            targets = self.preprocess(
                targets.to(self.device), batch_size, scale_tensor=imgsz[[1, 0, 1, 0]]
            )
            gt_labels, gt_bboxes = targets.split((1, 4), 2)
            mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0.0)

            pred_bboxes = self.bbox_decode(anchor_points, pred_distri)

            _, target_bboxes, target_scores, fg_mask, target_gt_idx = self.assigner(
                pred_scores.detach().sigmoid(),
                (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
                anchor_points * stride_tensor,
                gt_labels,
                gt_bboxes,
                mask_gt,
            )

            w_cls, w_seg = compute_anchor_equalization_weights(
                fg_mask,
                target_gt_idx,
                target_scores,
                topk=self.tal_topk,
                max_boost=self.max_boost,
            )
            self._last_w_seg = w_seg

            target_scores_sum = max(target_scores.sum(), 1)

            bce_loss = self.bce(pred_scores, target_scores.to(dtype)) * w_cls.to(dtype).unsqueeze(
                -1
            )
            if self.class_weights is not None:
                bce_loss = bce_loss * self.class_weights
            loss[1] = bce_loss.sum() / target_scores_sum

            if fg_mask.sum():
                weighted_target_scores = target_scores * w_cls.to(target_scores.dtype).unsqueeze(-1)
                loss[0], loss[2] = self.bbox_loss(
                    pred_distri,
                    pred_bboxes,
                    anchor_points,
                    target_bboxes / stride_tensor,
                    weighted_target_scores,
                    target_scores_sum,
                    fg_mask,
                    imgsz,
                    stride_tensor,
                )

            loss[0] *= self.hyp.box
            loss[1] *= self.hyp.cls
            loss[2] *= self.hyp.dfl
            return (
                (fg_mask, target_gt_idx, target_bboxes, anchor_points, stride_tensor),
                loss,
                dict(zip(self.loss_names, loss.detach())),
            )

        def calculate_segmentation_loss(
            self,
            fg_mask: torch.Tensor,
            masks: torch.Tensor,
            target_gt_idx: torch.Tensor,
            target_bboxes: torch.Tensor,
            batch_idx: torch.Tensor,
            proto: torch.Tensor,
            pred_masks: torch.Tensor,
            imgsz: torch.Tensor,
        ) -> torch.Tensor:
            _, _, mask_h, mask_w = proto.shape
            loss = 0
            target_bboxes_normalized = target_bboxes / imgsz[[1, 0, 1, 0]]
            marea = xyxy2xywh(target_bboxes_normalized)[..., 2:].prod(2)
            mxyxy = target_bboxes_normalized * torch.tensor(
                [mask_w, mask_h, mask_w, mask_h], device=proto.device
            )
            w_seg = (
                self._last_w_seg
                if self._last_w_seg is not None and self._last_w_seg.shape == fg_mask.shape
                else torch.ones_like(fg_mask, dtype=torch.float32)
            )

            for i, single_i in enumerate(
                zip(fg_mask, target_gt_idx, pred_masks, proto, mxyxy, marea, w_seg)
            ):
                fg_mask_i, target_gt_idx_i, pred_masks_i, proto_i, mxyxy_i, marea_i, w_seg_i = (
                    single_i
                )
                if fg_mask_i.any():
                    mask_idx = target_gt_idx_i[fg_mask_i]
                    if self.overlap:
                        gt_mask = (masks[i] == (mask_idx + 1).view(-1, 1, 1)).float()
                    else:
                        gt_mask = masks[batch_idx.view(-1) == i][mask_idx]

                    eff_area = marea_i[fg_mask_i] / w_seg_i[fg_mask_i].to(marea_i.dtype).clamp(
                        min=1e-6
                    )
                    loss += self.single_mask_loss(
                        gt_mask,
                        pred_masks_i[fg_mask_i],
                        proto_i,
                        mxyxy_i[fg_mask_i],
                        eff_area,
                    )
                else:
                    loss += (proto * 0).sum() + (pred_masks * 0).sum()

            return loss / fg_mask.sum()

    class SmallTALSegmentationTrainer(base_trainer_cls):
        def set_model_attributes(self):
            super().set_model_attributes()
            umodel = unwrap_model(self.model)
            umodel.nc = 1
            umodel.names = {0: "filament"}
            umodel.criterion = SmallTALSegmentationLoss(umodel)

    return base_ds_cls, SmallTALSegmentationLoss, SmallTALSegmentationTrainer


def build_pure_small_tal_trainer_classes(*, max_boost: float = 4.0):
    """Build standard ``single_cls=True`` Ultralytics Loss & Trainer with Small-TAL equalization.

    Leaves ``YOLODataset``, ``data.yaml`` (``nc=1``), and RNG/transform streams 100% identical
    to ``prod`` (``yolo_fold_{0,1,2}`` and ``v96`` ``yolo_full_fold2``), changing only
    ``umodel.criterion``.
    """
    from ultralytics.models.yolo.segment import SegmentationTrainer
    from ultralytics.utils.loss import v8SegmentationLoss
    from ultralytics.utils.torch_utils import unwrap_model

    _, spine_tal_loss_cls, _ = build_small_tal_trainer_classes(
        max_boost=max_boost,
        aux_weight=0.0,
    )

    class PureSmallTALSegmentationLoss(v8SegmentationLoss):
        def __init__(self, model, tal_topk: int = 10):
            super().__init__(model, tal_topk=tal_topk)
            self.tal_topk = int(tal_topk)
            self.max_boost = float(max_boost)
            self._last_w_seg: torch.Tensor | None = None

        get_assigned_targets_and_loss = spine_tal_loss_cls.get_assigned_targets_and_loss
        calculate_segmentation_loss = spine_tal_loss_cls.calculate_segmentation_loss

    class PureSmallTALSegmentationTrainer(SegmentationTrainer):
        def set_model_attributes(self):
            super().set_model_attributes()
            umodel = unwrap_model(self.model)
            umodel.criterion = PureSmallTALSegmentationLoss(umodel)

    return PureSmallTALSegmentationLoss, PureSmallTALSegmentationTrainer
# --- END small_tal_yolo.py ---

# --- BEGIN consensus_fusion.py ---
"""1-to-1 Multi-detector bipartite matching and continuous score/mask fusion (Idea 8, v132).

Fixes three structural issues in ``run_fold_oof_eval_consensus.py`` (v120/v121c/v124c):
1. Replaces unconstrained transitive union-find with greedy descending-IoU 1-to-1 bipartite
   clustering (at most 1 proposal per detector in every cluster), preventing adjacent-filament
   chaining across detectors.
2. Replaces hard step-function vote counting (``count(s >= 0.26) >= 2``, which discards
   ``(0.45, 0.24, 0)``, ``(0.24, 0.24, 0.24)``, and solo ``(0.75, 0, 0)``) with continuous
   calibrated cluster scoring ``sum_d w_d * s_d`` (missing detector contributes ``0.0``).
3. Supports score-weighted multi-detector proposal mask fusion before ``MediSegEdgeAttUNet``.
"""


from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_utils


@dataclass(frozen=True)
class DetectorProposal:
    det_name: str
    conf: float
    raw_rle: dict
    prop_idx: int


@dataclass
class MatchedCluster:
    members: dict[str, DetectorProposal]  # det_name -> DetectorProposal (strictly <=1 per det)

    def score_of(self, det_name: str) -> float:
        p = self.members.get(det_name)
        return float(p.conf) if p is not None else 0.0

    def weighted_score(self, weights: dict[str, float]) -> float:
        return float(sum(w * self.score_of(d) for d, w in weights.items()))

    def max_conf(self) -> float:
        return max((float(p.conf) for p in self.members.values()), default=0.0)

    def count_above(self, thr: float) -> int:
        return sum(1 for p in self.members.values() if float(p.conf) >= thr)

    def best_member(self) -> DetectorProposal:
        return max(self.members.values(), key=lambda p: (float(p.conf), p.det_name))


def match_detector_proposals_1to1(
    proposals: list[DetectorProposal],
    *,
    iou_threshold: float = 0.45,
) -> list[MatchedCluster]:
    """Group cross-detector proposals by descending mask IoU with <=1 member per detector."""
    n = len(proposals)
    if n == 0:
        return []
    rles = [p.raw_rle for p in proposals]
    iou_mat = np.asarray(mask_utils.iou(rles, rles, [0] * n), dtype=np.float64)

    parent = list(range(n))
    det_sets = [{proposals[i].det_name} for i in range(n)]

    def find(x: int) -> int:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    pairs: list[tuple[float, int, int]] = []
    for i in range(n):
        di = proposals[i].det_name
        for j in range(i + 1, n):
            if di == proposals[j].det_name:
                continue
            v = float(iou_mat[i, j])
            if v >= iou_threshold:
                pairs.append((v, i, j))
    pairs.sort(key=lambda t: (-t[0], t[1], t[2]))

    for _, i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj and det_sets[ri].isdisjoint(det_sets[rj]):
            parent[rj] = ri
            det_sets[ri] |= det_sets[rj]

    grouped: dict[int, dict[str, DetectorProposal]] = {}
    for idx in range(n):
        r = find(idx)
        p = proposals[idx]
        grouped.setdefault(r, {})[p.det_name] = p

    clusters = [MatchedCluster(members=m) for m in grouped.values()]
    clusters.sort(key=lambda c: -c.max_conf())
    return clusters


def fuse_cluster_proposal_mask(
    cluster: MatchedCluster,
    *,
    mode: str = "max_conf",
    vote_threshold: float = 0.45,
) -> np.ndarray:
    """Build bool proposal mask for ``MediSegEdgeAttUNet`` from a matched cluster."""
    if mode == "max_conf" or len(cluster.members) == 1:
        best = cluster.best_member()
        return mask_utils.decode(best.raw_rle) != 0
    if mode != "soft_vote":
        raise ValueError(f"Unknown mask fusion mode: {mode}")

    acc: np.ndarray | None = None
    w_sum = 0.0
    for p in cluster.members.values():
        w = max(float(p.conf), 1e-4)
        m = (mask_utils.decode(p.raw_rle) != 0).astype(np.float32)
        acc = (w * m) if acc is None else (acc + w * m)
        w_sum += w
    assert acc is not None
    return (acc / max(w_sum, 1e-6)) >= float(vote_threshold)
# --- END consensus_fusion.py ---

# max_boost=1.0 と標準 v8SegmentationLoss の数値一致検査 (FP32)
from ultralytics.cfg import get_cfg as _get_cfg
from ultralytics.nn.tasks import SegmentationModel as _SegModel
from ultralytics.utils.loss import v8SegmentationLoss as _v8SegLoss

torch.manual_seed(42)
_m = _SegModel("yolo11n-seg.yaml", ch=3, nc=1, verbose=False)
_m.args = _get_cfg(overrides={"task": "segment", "overlap_mask": False})
_lv8 = _v8SegLoss(_m)
_lp1_cls, _ = build_pure_small_tal_trainer_classes(max_boost=1.0)
_lp1 = _lp1_cls(_m)
_x = torch.randn(2, 3, 128, 128)
_pr = _m(_x)
_mk = torch.zeros((2, 32, 32), dtype=torch.float32); _mk[0, 4:18, 4:20] = 1.0; _mk[1, 22:25, 22:25] = 1.0
_bt = {"img": _x, "batch_idx": torch.tensor([0.0, 1.0]), "cls": torch.tensor([[0.0], [0.0]]),
       "bboxes": torch.tensor([[0.375, 0.34375, 0.5, 0.4375], [0.734375, 0.734375, 0.09375, 0.09375]]), "masks": _mk}
_tb, _ = _lv8(_pr, _bt); _t1, _ = _lp1(_pr, _bt)
PARITY_MAX_DIFF = float((_tb - _t1).abs().max().item())
assert PARITY_MAX_DIFF < 1e-5, f"Parity check failed: {PARITY_MAX_DIFF}"
print(f"[Sanity] PureSmallTAL(max_boost=1.0) vs v8SegmentationLoss max_diff={PARITY_MAX_DIFF:.2e} OK")
del _m, _lv8, _lp1, _x, _pr, _mk, _bt, _tb, _t1

def polygon_line(ann, width, height):
    polygons = [p for p in ann.get("segmentation", []) if len(p) >= 6]
    if not polygons:
        return None
    polygon = max(polygons, key=len)
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    points[:, 0] = np.clip(points[:, 0] / width, 0, 1)
    points[:, 1] = np.clip(points[:, 1] / height, 0, 1)
    return "0 " + " ".join(f"{value:.6f}" for value in points.reshape(-1))

def write_variant(fold_root, split, file_name, image_id, suffix):
    info = images[image_id]
    stem = Path(file_name).stem + suffix
    image_out = fold_root / "images" / split / f"{stem}.jpeg"
    label_out = fold_root / "labels" / split / f"{stem}.txt"
    image_out.parent.mkdir(parents=True, exist_ok=True)
    if not image_out.exists():
        shutil.copy2((CFG.PREPROC_DIR / file_name).resolve(), image_out)
    lines = [polygon_line(ann, info["width"], info["height"])
             for ann in annotations[image_id] if not ann.get("iscrowd", 0)]
    label_out.parent.mkdir(parents=True, exist_ok=True)
    label_out.write_text("\n".join(line for line in lines if line), encoding="utf-8")

def build_fold_dataset(fold):
    root = CFG.YOLO_ROOT / f"val{fold}_train{''.join(map(str, CFG.TRAIN_FOLDS))}"
    reference_id = {name: file_to_ids[name][0] for name in train_files}
    stats = {"train_samples": 0, "train_instances": 0, "val_samples": 0, "val_instances": 0}
    for file_name in tqdm(train_files, desc=f"fold {fold} データセット構築 (local SSD)"):
        if fold_of_file[file_name] == fold:
            write_variant(root, "val", file_name, reference_id[file_name], "")
            stats["val_samples"] += 1
            stats["val_instances"] += len(annotations[reference_id[file_name]])
        if fold_of_file[file_name] in CFG.TRAIN_FOLDS:
            for image_id in file_to_ids[file_name]:
                write_variant(root, "train", file_name, image_id, f"__labeler_{image_id}")
                stats["train_samples"] += 1
                stats["train_instances"] += len(annotations[image_id])
    yaml_path = root / "data.yaml"
    yaml_path.write_text(f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
                         encoding="utf-8")
    return yaml_path, stats

_, PureSmallTALTrainer = build_pure_small_tal_trainer_classes(max_boost=CFG.MAX_BOOST)

def train_arm(arm):
    run_name = f"yolo_{arm}_fold{CFG.FOLD}"
    run_dir = CFG.RUNS_ROOT / run_name
    best_path, last_path, done_path = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt", run_dir / "TRAINING_COMPLETE"
    yaml_path, stats = build_fold_dataset(CFG.FOLD)
    if done_path.exists() and best_path.exists():
        print(f"{arm}: 学習済み checkpoint を再利用 {best_path}")
        return best_path, stats
    started = time.time()
    if last_path.exists() and not done_path.exists():
        print(f"{arm}: 中断 last.pt から PureSmallTALTrainer で再開 {last_path}")
        YOLO(str(last_path)).train(resume=True, trainer=PureSmallTALTrainer)
        stats["resumed_from_last_pt"] = True
    else:
        YOLO(CFG.YOLO_MODEL).train(
            trainer=PureSmallTALTrainer,
            data=str(yaml_path), epochs=CFG.YOLO_EPOCHS, imgsz=CFG.YOLO_IMGSZ,
            batch=CFG.YOLO_BATCH, device=0, workers=4, patience=CFG.YOLO_PATIENCE,
            seed=CFG.SEED + CFG.FOLD, deterministic=True, single_cls=True,
            mask_ratio=CFG.YOLO_MASK_RATIO, overlap_mask=False, amp=True, cache="disk",
            cls=CFG.YOLO_CLS,
            optimizer="AdamW", lr0=7e-4, lrf=0.05, weight_decay=5e-4, warmup_epochs=3,
            degrees=180, translate=0.04, scale=0.25, fliplr=0.5, flipud=0.5,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.18, mosaic=0.20, close_mosaic=10,
            project=str(CFG.RUNS_ROOT), name=run_name, exist_ok=True,
        )
    stats["train_minutes"] = round((time.time() - started) / 60, 1)
    done_path.write_text("ok", encoding="utf-8")
    return best_path, stats


In [ ]:
# [4/6] refiner モデル定義 (solar_filament/models/mediseg_models.py と同一)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


In [ ]:
# [5/6] 評価関数 (prod 完全準拠 + バッチ化高速 Refiner + paired bootstrap + 1-to-1 融合)
def sparse_label_pq(sum_iou, tp, fp, fn, r=0.77):
    denominator = r * tp + 0.5 * fp + 0.5 * (1 - r) * tp + 0.5 * r * fn
    return float(r * sum_iou / denominator) if denominator > 0 else 0.0

refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
_ckpt = torch.load(CFG.REFINER_CKPT, map_location="cuda", weights_only=False)
refiner.load_state_dict(_ckpt.get("best_model") or _ckpt["model"], strict=True)
del _ckpt
REFINER_SHA256 = file_sha256(CFG.REFINER_CKPT)

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    out = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        out.append(mask >= 0.5)
    conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return out, conf

def mask_square_box(mask, context=CFG.CROP_CONTEXT, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposals_batched(image, proposals, batch_size=32):
    if not proposals:
        return []
    boxes = [mask_square_box(p) for p in proposals]
    crops = [cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
             for (x0, y0, x1, y1) in boxes]
    probs_all = []
    for s in range(0, len(crops), batch_size):
        arr = np.stack([c.transpose(2, 0, 1) for c in crops[s:s + batch_size]], axis=0)
        ten = torch.from_numpy(arr).float().cuda() / 255.0
        probs_all.append(refiner(ten).sigmoid()[:, 0].cpu().numpy())
    probs = np.concatenate(probs_all, axis=0)
    out = []
    for pmask, (x0, y0, x1, y1), pr in zip(proposals, boxes, probs):
        pr_up = cv2.resize(pr, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
        ref = np.zeros_like(pmask, dtype=bool)
        ref[y0:y1, x0:x1] = pr_up >= CFG.REFINER_THRESHOLD
        out.append(ref & pmask)
    return out

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped or self_iou[i, j] <= 0:
                continue
            inter = self_iou[i, j] * (areas[i] + areas[j]) / (1.0 + self_iou[i, j])
            smaller = i if areas[i] <= areas[j] else j
            if inter / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    occupied = np.zeros(masks[0].shape, dtype=bool)
    kept = []
    for index in sorted(drop_contained_indices(masks), key=lambda i: areas[i]):
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < CFG.MIN_AREA:
            continue
        kept.append(exclusive); occupied |= exclusive
    return kept

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

def build_detector_cache(weights_path, label):
    model = YOLO(str(weights_path))
    cache = []
    for name in tqdm(valid_files, desc=f"{label} 推論+refine"):
        path = CFG.PREPROC_DIR / name
        result = model.predict(str(path), imgsz=CFG.YOLO_IMGSZ, conf=CFG.PREDICT_CONF_FLOOR,
                               iou=CFG.PREDICT_IOU, agnostic_nms=True, retina_masks=True, verbose=False)[0]
        masks, conf = result_masks(result)
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        valid_idx = [i for i, m in enumerate(masks) if float(m.sum()) >= CFG.MIN_AREA]
        vmasks = [masks[i] for i in valid_idx]
        vconf = conf[valid_idx] if len(valid_idx) else np.zeros(0, np.float32)
        ref_masks = refine_proposals_batched(image, vmasks)
        raw_rles = [mask_utils.encode(np.asfortranarray(m.astype(np.uint8))) for m in vmasks]
        ref_rles = [mask_utils.encode(np.asfortranarray(rm.astype(np.uint8))) for rm in ref_masks]
        cache.append((name, vconf, [float(m.sum()) for m in vmasks], raw_rles, ref_rles))
    del model; torch.cuda.empty_cache()
    return cache

def score_cache(cache):
    out, per_obs_026 = {}, {}
    for c in CFG.CONF_SWEEP:
        sum_iou = 0.0; tp = 0; fp = 0; fn = 0; n_pred = 0
        obs_dict = {}
        for name, conf, areas, raw_rles, refined in cache:
            keep = [i for i in range(len(conf)) if conf[i] >= c and areas[i] >= CFG.MIN_AREA]
            preds = apply_ownership([mask_utils.decode(refined[i]) != 0 for i in keep])
            pred_rles = mask_utils.encode(np.asfortranarray(np.stack(preds, -1).astype(np.uint8))) if preds else []
            n_pred += len(pred_rles)
            os_s = [0.0, 0, 0, 0]
            for image_id in file_to_ids[name]:
                truth = gt_rles[image_id]
                mat = iou_matrix(truth, pred_rles)
                if len(truth) == 0:
                    fp += len(pred_rles); os_s[2] += len(pred_rles)
                elif len(pred_rles) == 0:
                    fn += len(truth); os_s[3] += len(truth)
                else:
                    hits = mat > 0.5
                    s_i = float(mat[hits].sum()); t_i = int(hits.sum())
                    fp_i = int((hits.sum(axis=0) == 0).sum()); fn_i = int((hits.sum(axis=1) == 0).sum())
                    sum_iou += s_i; tp += t_i; fp += fp_i; fn += fn_i
                    os_s[0] += s_i; os_s[1] += t_i; os_s[2] += fp_i; os_s[3] += fn_i
            obs_dict[name] = tuple(os_s)
        den = tp + 0.5 * fp + 0.5 * fn
        pq = sum_iou / den if den > 0 else 0.0
        out[f"{c:.2f}"] = {
            "pq": float(pq), "sq": float(sum_iou / tp) if tp else 0.0, "rq": float(tp / den) if den > 0 else 0.0,
            "tp": tp, "fp": fp, "fn": fn,
            "pq_sparse_r077": sparse_label_pq(sum_iou, tp, fp, fn, CFG.SPARSE_R),
            "preds_per_image": n_pred / max(len(valid_files), 1),
        }
        if abs(c - CFG.PRIMARY_CONF) < 1e-6:
            per_obs_026 = obs_dict
    return out, per_obs_026

def score_2det_fusion(cache_prod, cache_tal, thr_list=(0.18, 0.20, 0.22, 0.24, 0.26)):
    out = {}
    obs_best = {}
    best_pq = -1.0; best_key = None
    for thr in thr_list:
        sum_iou = 0.0; tp = 0; fp = 0; fn = 0; obs_d = {}
        for (name, c_p, a_p, raw_p, ref_p), (_, c_t, a_t, raw_t, ref_t) in zip(cache_prod, cache_tal):
            props = []
            for i in range(len(c_p)):
                if a_p[i] >= CFG.MIN_AREA:
                    props.append(DetectorProposal("prod", float(c_p[i]), raw_p[i], i))
            for j in range(len(c_t)):
                if a_t[j] >= CFG.MIN_AREA:
                    props.append(DetectorProposal("tal", float(c_t[j]), raw_t[j], j))
            clusters = match_detector_proposals_1to1(props, iou_threshold=0.45)
            chosen = []
            for cl in clusters:
                sc = 0.5 * cl.score_of("prod") + 0.5 * cl.score_of("tal")
                if sc >= thr:
                    bm = cl.best_member()
                    rr = ref_p[bm.prop_idx] if bm.det_name == "prod" else ref_t[bm.prop_idx]
                    chosen.append(mask_utils.decode(rr) != 0)
            preds = apply_ownership(chosen)
            pred_rles = mask_utils.encode(np.asfortranarray(np.stack(preds, -1).astype(np.uint8))) if preds else []
            os_s = [0.0, 0, 0, 0]
            for image_id in file_to_ids[name]:
                truth = gt_rles[image_id]
                mat = iou_matrix(truth, pred_rles)
                if len(truth) == 0:
                    fp += len(pred_rles); os_s[2] += len(pred_rles)
                elif len(pred_rles) == 0:
                    fn += len(truth); os_s[3] += len(truth)
                else:
                    hits = mat > 0.5
                    s_i = float(mat[hits].sum()); t_i = int(hits.sum())
                    fp_i = int((hits.sum(axis=0) == 0).sum()); fn_i = int((hits.sum(axis=1) == 0).sum())
                    sum_iou += s_i; tp += t_i; fp += fp_i; fn += fn_i
                    os_s[0] += s_i; os_s[1] += t_i; os_s[2] += fp_i; os_s[3] += fn_i
            obs_d[name] = tuple(os_s)
        den = tp + 0.5 * fp + 0.5 * fn
        pq = sum_iou / den if den > 0 else 0.0
        k = f"mean2_{thr:.2f}"
        out[k] = {"pq": float(pq), "sq": float(sum_iou / tp) if tp else 0.0, "rq": float(tp / den) if den > 0 else 0.0,
                  "tp": tp, "fp": fp, "fn": fn, "pq_sparse_r077": sparse_label_pq(sum_iou, tp, fp, fn, CFG.SPARSE_R)}
        if pq > best_pq:
            best_pq = pq; best_key = k; obs_best = obs_d
    return out, best_key, obs_best

def paired_bootstrap_obs(obs_base, obs_cand, files, n_boot=2000, seed=3407):
    rng = np.random.default_rng(seed)
    ba = np.array([obs_base[f] for f in files], dtype=np.float64)
    ca = np.array([obs_cand[f] for f in files], dtype=np.float64)
    n = len(files)
    idx = rng.integers(0, n, size=(n_boot, n))
    bs = ba[idx].sum(axis=1); cs = ca[idx].sum(axis=1)
    bpq = np.where((bs[:, 1] + 0.5 * bs[:, 2] + 0.5 * bs[:, 3]) > 0, bs[:, 0] / (bs[:, 1] + 0.5 * bs[:, 2] + 0.5 * bs[:, 3]), 0.0)
    cpq = np.where((cs[:, 1] + 0.5 * cs[:, 2] + 0.5 * cs[:, 3]) > 0, cs[:, 0] / (cs[:, 1] + 0.5 * cs[:, 2] + 0.5 * cs[:, 3]), 0.0)
    d = cpq - bpq
    return {"n_obs": n, "mean_delta": float(d.mean()), "ci95_low": float(np.percentile(d, 2.5)),
            "ci95_high": float(np.percentile(d, 97.5)), "prob_positive": float((d > 0).mean())}


In [ ]:
# [6/6] 実行: prod baseline 検証 → Small-TAL 学習 → best/last + 2検出器融合評価 → manifest 保存
def save_manifest(manifest):
    CFG.MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=CFG.MANIFEST.parent, suffix=".tmp", delete=False) as h:
        h.write(payload); h.flush(); os.fsync(h.fileno()); tmp = h.name
    os.replace(tmp, CFG.MANIFEST)

cache_prod = build_detector_cache(CFG.PROD_YOLO, "prod")
prod_by_conf, obs_prod_026 = score_cache(cache_prod)
prod_pq = prod_by_conf["0.26"]["pq"]
print({"prod@0.26": round(prod_pq, 6), "既知値": CFG.BASELINE_PQ, "差": round(prod_pq - CFG.BASELINE_PQ, 6)})

arm = CFG.ARMS[0]
best_w, ds_stats = train_arm(arm)
last_w = Path(best_w).parent / "last.pt"

cache_best = build_detector_cache(best_w, arm + "_best")
best_by_conf, obs_best_026 = score_cache(cache_best)
boot_best = paired_bootstrap_obs(obs_prod_026, obs_best_026, valid_files, n_boot=CFG.BOOTSTRAP_ITERS, seed=CFG.SEED)
fus_best, fus_best_key, obs_fus_best = score_2det_fusion(cache_prod, cache_best)
boot_fus_best = paired_bootstrap_obs(obs_prod_026, obs_fus_best, valid_files, n_boot=CFG.BOOTSTRAP_ITERS, seed=CFG.SEED)

last_res = {}
if last_w.exists():
    cache_last = build_detector_cache(last_w, arm + "_last")
    last_by_conf, obs_last_026 = score_cache(cache_last)
    boot_last = paired_bootstrap_obs(obs_prod_026, obs_last_026, valid_files, n_boot=CFG.BOOTSTRAP_ITERS, seed=CFG.SEED)
    fus_last, fus_last_key, obs_fus_last = score_2det_fusion(cache_prod, cache_last)
    boot_fus_last = paired_bootstrap_obs(obs_prod_026, obs_fus_last, valid_files, n_boot=CFG.BOOTSTRAP_ITERS, seed=CFG.SEED)
    last_res = {
        "weights": str(last_w),
        "weights_sha256": file_sha256(last_w),
        "by_conf": last_by_conf,
        "delta_026_vs_prod": last_by_conf["0.26"]["pq"] - prod_pq,
        "bootstrap_vs_prod_026": boot_last,
        "fusion_2det_with_prod": fus_last,
        "fusion_best_key": fus_last_key,
        "bootstrap_fusion_vs_prod": boot_fus_last,
    }

# Honest OOF 時は v132 4検出器融合用にキャッシュも保存
if not CFG.IS_ALL_DATA:
    (CFG.CACHE_OUT_DIR / f"small_tal_f{CFG.FOLD}_best_cache.pkl").write_bytes(pickle.dumps(cache_best))
    if last_w.exists():
        (CFG.CACHE_OUT_DIR / f"small_tal_f{CFG.FOLD}_last_cache.pkl").write_bytes(pickle.dumps(cache_last))

manifest = {
    "experiment_id": 'v133b_small_tal_f2',
    "is_all_data_in_sample": CFG.IS_ALL_DATA,
    "fold": CFG.FOLD,
    "train_folds": list(CFG.TRAIN_FOLDS),
    "max_boost": CFG.MAX_BOOST,
    "parity_max_diff": PARITY_MAX_DIFF,
    "runtime": RUNTIME_INFO,
    "hashes": {
        "annotations_sha256": file_sha256(TRAIN_JSON),
        "folds_sha256": folds_sha256,
        "refiner_sha256": REFINER_SHA256,
        "prod_yolo_sha256": file_sha256(CFG.PROD_YOLO),
    },
    "dataset_stats": ds_stats,
    "prod": {"by_conf": prod_by_conf},
    "small_tal_best": {
        "weights": str(best_w),
        "weights_sha256": file_sha256(best_w),
        "by_conf": best_by_conf,
        "delta_026_vs_prod": best_by_conf["0.26"]["pq"] - prod_pq,
        "bootstrap_vs_prod_026": boot_best,
        "fusion_2det_with_prod": fus_best,
        "fusion_best_key": fus_best_key,
        "bootstrap_fusion_vs_prod": boot_fus_best,
    },
    "small_tal_last": last_res,
}
save_manifest(manifest)

print("=== v133b_small_tal_f2 SUMMARY (in_sample=" + str(CFG.IS_ALL_DATA) + ") ===")
print(f"prod (best.pt @0.26)      | PQ={prod_pq:.5f} | sparse={prod_by_conf['0.26']['pq_sparse_r077']:.5f} | TP={prod_by_conf['0.26']['tp']} FP={prod_by_conf['0.26']['fp']} FN={prod_by_conf['0.26']['fn']}")
print(f"small_tal best.pt (@0.26) | PQ={best_by_conf['0.26']['pq']:.5f} ({best_by_conf['0.26']['pq']-prod_pq:+.5f}) | CI95=[{boot_best['ci95_low']:+.5f}, {boot_best['ci95_high']:+.5f}] P(>0)={boot_best['prob_positive']:.4f}")
if last_res:
    lpq = last_res["by_conf"]["0.26"]["pq"]
    bl = last_res["bootstrap_vs_prod_026"]
    print(f"small_tal last.pt (@0.26) | PQ={lpq:.5f} ({lpq-prod_pq:+.5f}) | CI95=[{bl['ci95_low']:+.5f}, {bl['ci95_high']:+.5f}] P(>0)={bl['prob_positive']:.4f}")
print(f"2-det fusion (prod+best)  | key={fus_best_key} PQ={fus_best[fus_best_key]['pq']:.5f} ({fus_best[fus_best_key]['pq']-prod_pq:+.5f}) | CI95=[{boot_fus_best['ci95_low']:+.5f}, {boot_fus_best['ci95_high']:+.5f}]")
if last_res:
    fk = last_res["fusion_best_key"]
    fpq = last_res["fusion_2det_with_prod"][fk]["pq"]
    bfl = last_res["bootstrap_fusion_vs_prod"]
    print(f"2-det fusion (prod+last)  | key={fk} PQ={fpq:.5f} ({fpq-prod_pq:+.5f}) | CI95=[{bfl['ci95_low']:+.5f}, {bfl['ci95_high']:+.5f}]")
print("Saved manifest:", CFG.MANIFEST)
